# Summarize private docs 'Txt' with Ollama using LangChain's PromptTemplate


This notebook steps to building a practical application that can read and summarize documents. 

## Setup

In [1]:
%%time
import warnings
warnings.filterwarnings("ignore")

!python3 -m pip install --upgrade pip

CPU times: user 11.9 ms, sys: 10.6 ms, total: 22.6 ms
Wall time: 1.38 s


### OSS libraries install

In [2]:
%pip install sentence_transformers chromadb wget huggingface langchain langchain_chroma langchain-community langchain-huggingface unstructured langchain-ollama ipython-autotime --use-deprecated=legacy-resolver

Note: you may need to restart the kernel to use updated packages.


### My variables

In [3]:
my_model_ollama = "llama3.2"

## Preprocessing
### Load the document

The document, which is provided in a TXT format (./docs/CompanyPolicies.txt), outlines some company policies and serves as an example data set for the project.

In [4]:
filenames = ["./docs/CompanyPolicies.txt", "./docs/StateOfUnion.txt"]

for filename in filenames:
    if len(filename) > 0:
        print(f"\n **** Loading from {filename} \n\n")
        with open(filename, "r") as file:
            # Read the contents of the file
            contents = file.read()
            print(contents)
    else:
        print(f"Failed to retrieve content from {filename}")
    print(f"\n-------------------------------- \n\n")


 **** Loading from ./docs/CompanyPolicies.txt 


1.	Code of Conduct

Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.
Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.
Respect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.
Accountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continuously improve our

### Splitting the document into chunks

In [5]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter

filename = "./docs/CompanyPolicies.txt"
loader = TextLoader(filename)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
print(len(texts))

Created a chunk of size 1624, which is longer than the specified 1000
Created a chunk of size 1885, which is longer than the specified 1000
Created a chunk of size 1903, which is longer than the specified 1000
Created a chunk of size 1729, which is longer than the specified 1000
Created a chunk of size 1678, which is longer than the specified 1000
Created a chunk of size 2032, which is longer than the specified 1000
Created a chunk of size 1894, which is longer than the specified 1000


16


### Embedding and storing

In [6]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma


embeddings = HuggingFaceEmbeddings()
docsearch = Chroma.from_documents(
    texts, embeddings
)  # store the embedding in docsearch using Chromadb
print("document ingested")

/var/folders/7w/gt6dll1d15163pf4zh70djxw0000gp/T/ipykernel_88905/1539368597.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/var/folders/7w/gt6dll1d15163pf4zh70djxw0000gp/T/ipykernel_88905/1539368597.py:5: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


document ingested


## LLM model construction

In [7]:
from langchain_ollama.llms import OllamaLLM

llm_client = OllamaLLM(
    model=my_model_ollama,
    base_url="http://localhost:11434",
    headers={"Content-Type": "application/json"},
    stream=True,
)

llm_client

OllamaLLM(model='llama3.2', base_url='http://localhost:11434')

## Integrating LangChain

In [8]:
from langchain.chains import RetrievalQA

show_source_documents = False

retriever_qa = RetrievalQA.from_chain_type(
    llm=llm_client,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    return_source_documents=show_source_documents,
)
retriever_qa

RetrievalQA(verbose=False, combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n{context}\n\nQuestion: {question}\nHelpful Answer:"), llm=OllamaLLM(model='llama3.2', base_url='http://localhost:11434'), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='{page_content}'), document_variable_name='context'), retriever=VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x117f46390>, search_kwargs={}))

In [9]:
query = "what is mobile policy?"
retriever_qa.invoke(query)

{'query': 'what is mobile policy?',
 'result': 'The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization, aiming to promote secure and compliant use of mobile devices in line with legal and ethical standards.'}

In [10]:
query = "Can you summarize the document for me?"
retriever_qa.invoke(query)

{'query': 'Can you summarize the document for me?',
 'result': "I don't know how to summarize the entire document, as it appears to be a multi-page policy document outlining various aspects of an organization's Code of Conduct and Health and Safety policies. However, I can try to provide a general overview of the document.\n\nThe document outlines the organization's commitment to integrity, respect, accountability, safety, environmental responsibility, health and safety, and anti-discrimination and harassment policies. It emphasizes the importance of upholding these principles in all interactions and behaviors, and encourages employees to serve as role models for others.\n\nIf you'd like more specific information or details from a particular section of the document, I can try to help with that."}

In [11]:
query = "Can I eat in company vehicles?"
retriever_qa.invoke(query)

{'query': 'Can I eat in company vehicles?',
 'result': "I don't know, and the provided policies do not specifically address eating in company vehicles. The No Smoking in Company Vehicles policy only prohibits smoking in company vehicles, but does not mention food or drink consumption."}

### Using prompt template

In [12]:
from langchain.prompts import PromptTemplate

prompt_template = """Use the information from the document to answer the question at the end. If you don't know the answer, just say that you don't know, definately do not try to make up an answer.

{context}

Question: {question}
"""

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

chain_type_kwargs = {"prompt": PROMPT}

In [13]:
retriever_qa = RetrievalQA.from_chain_type(
    llm=llm_client,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    chain_type_kwargs=chain_type_kwargs,
    return_source_documents=False,
)

# retriever_qa.chain_type_kwargs = chain_type_kwargs

retriever_qa

RetrievalQA(verbose=False, combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the information from the document to answer the question at the end. If you don't know the answer, just say that you don't know, definately do not try to make up an answer.\n\n{context}\n\nQuestion: {question}\n"), llm=OllamaLLM(model='llama3.2', base_url='http://localhost:11434'), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='{page_content}'), document_variable_name='context'), retriever=VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x117f46390>, search_kwargs={}))

In [14]:
query = "Can I eat in company vehicles?"
retriever_qa.invoke(query)

{'query': 'Can I eat in company vehicles?',
 'result': "I don't know. The provided document doesn't mention eating in company vehicles. It only mentions that smoking is not permitted in company vehicles, but it does not address food consumption."}

### Make the conversation have memory

In [15]:
query = "What I cannot do in it?"
retriever_qa.invoke(query)

{'query': 'What I cannot do in it?',
 'result': 'Based on the document, it appears that you cannot use company-provided internet and email services for non-work-related tasks during work hours (section 3).'}

#### To make the LLM have memory, you introduce the ConversationBufferMemory function from LangChain.

In [16]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key="chat_history", return_message=True)

memory

/var/folders/7w/gt6dll1d15163pf4zh70djxw0000gp/T/ipykernel_88905/2348225548.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_message=True)


ConversationBufferMemory(chat_memory=InMemoryChatMessageHistory(messages=[]), memory_key='chat_history')

In [17]:
from langchain.chains import ConversationalRetrievalChain

retriever_qa = ConversationalRetrievalChain.from_llm(
    llm=llm_client,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    memory=memory,
    get_chat_history=lambda h: h,
    return_source_documents=False,
)
retriever_qa

ConversationalRetrievalChain(memory=ConversationBufferMemory(chat_memory=InMemoryChatMessageHistory(messages=[]), memory_key='chat_history'), verbose=False, combine_docs_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n{context}\n\nQuestion: {question}\nHelpful Answer:"), llm=OllamaLLM(model='llama3.2', base_url='http://localhost:11434'), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='{page_content}'), document_variable_name='context'), question_generator=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['chat_history', 'question'], input_types={}, partial_variab

#### Create a `history` list to store the chat history. 

In [18]:
history = []

#### Create a `ConversationalRetrievalChain` to retrieve information and talk with the LLM.

In [19]:
from langchain.chains import ConversationalRetrievalChain

retriever_qa = ConversationalRetrievalChain.from_llm(
    llm=llm_client,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    memory=memory,
    get_chat_history=lambda h: h,
    return_source_documents=False,
)
retriever_qa

ConversationalRetrievalChain(memory=ConversationBufferMemory(chat_memory=InMemoryChatMessageHistory(messages=[]), memory_key='chat_history'), verbose=False, combine_docs_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n{context}\n\nQuestion: {question}\nHelpful Answer:"), llm=OllamaLLM(model='llama3.2', base_url='http://localhost:11434'), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='{page_content}'), document_variable_name='context'), question_generator=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['chat_history', 'question'], input_types={}, partial_variab

In [20]:
query = "What is mobile policy?"
result = retriever_qa.invoke({"question": query}, {"chat_history": history})
result

{'question': 'What is mobile policy?',
 'chat_history': '',
 'answer': 'The Mobile Phone Policy is a set of guidelines and expectations governing the responsible usage of mobile devices in the organization, aiming to promote secure use, legal compliance, and adherence to company values.'}

In [21]:
print(result["answer"])

The Mobile Phone Policy is a set of guidelines and expectations governing the responsible usage of mobile devices in the organization, aiming to promote secure use, legal compliance, and adherence to company values.


In [22]:
history.append((query, result["answer"]))
history

[('What is mobile policy?',
  'The Mobile Phone Policy is a set of guidelines and expectations governing the responsible usage of mobile devices in the organization, aiming to promote secure use, legal compliance, and adherence to company values.')]

In [23]:
query = "List points in it?"
result = retriever_qa({"question": query}, {"chat_history": history})
print(result["answer"])

/var/folders/7w/gt6dll1d15163pf4zh70djxw0000gp/T/ipykernel_88905/1087633897.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = retriever_qa({"question": query}, {"chat_history": history})


The Mobile Phone Policy covers the following specific points:

1. Acceptable Use: Mobile devices are primarily intended for work-related tasks, with limited personal usage allowed that does not disrupt work obligations.
2. Security: Safeguard mobile device and access credentials, exercise caution when downloading apps or clicking links from unfamiliar sources, and report security concerns or suspicious activities related to the mobile device.
3. Confidentiality: Avoid transmitting sensitive company information via unsecured messaging apps or emails, and be discreet when discussing company matters in public spaces.
4. Cost Management: Keep personal phone usage separate from company accounts and reimburse the company for any personal charges on company-issued phones.
5. Compliance: Adhere to all pertinent laws and regulations concerning mobile phone usage, including those related to data protection and privacy.
6. Lost or Stolen Devices: Immediately report lost or stolen mobile devices t

In [24]:
history.append((query, result["answer"]))
history

[('What is mobile policy?',
  'The Mobile Phone Policy is a set of guidelines and expectations governing the responsible usage of mobile devices in the organization, aiming to promote secure use, legal compliance, and adherence to company values.'),
 ('List points in it?',
  'The Mobile Phone Policy covers the following specific points:\n\n1. Acceptable Use: Mobile devices are primarily intended for work-related tasks, with limited personal usage allowed that does not disrupt work obligations.\n2. Security: Safeguard mobile device and access credentials, exercise caution when downloading apps or clicking links from unfamiliar sources, and report security concerns or suspicious activities related to the mobile device.\n3. Confidentiality: Avoid transmitting sensitive company information via unsecured messaging apps or emails, and be discreet when discussing company matters in public spaces.\n4. Cost Management: Keep personal phone usage separate from company accounts and reimburse the c

### Wrap up and make it an agent

In [25]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
import pprint


def agentFunction(input_string):

    memory = ConversationBufferMemory(memory_key="chat_history", return_message=True)
    qa = ConversationalRetrievalChain.from_llm(
        llm=llm_client,
        chain_type="stuff",
        retriever=docsearch.as_retriever(),
        memory=memory,
        get_chat_history=lambda h: h,
        return_source_documents=False,
    )
    history = []

    if (input_string is None) or (len(input_string) <= 0):
        input_string = input("Question: ")
    else:
        print("Question: ", input_string)

    result = qa({"question": input_string}, {"chat_history": history})

    history.append((query, result["answer"]))

    print("Answer: ", result["answer"])

    print("\n\n\n ------------ RESULT OBJECT ---------------\n")
    pprint.pprint(result)

In [26]:
agentFunction(
    "What is the smoking policy? Can you list all points of it? Can you summarize it?"
)

Question:  What is the smoking policy? Can you list all points of it? Can you summarize it?
Answer:  Based on the provided context, I can answer the questions as follows:

1. What is the smoking policy?

The smoking policy is a company-wide policy aimed at providing clear guidance and expectations concerning smoking on company premises.

2. List all points of the smoking policy:

Here are the points mentioned in the context:

* The Smoking Policy has been established to provide clear guidance and expectations concerning smoking on company premises.
* Smoking is only permitted in designated smoking areas, as marked by appropriate signage.
* These areas have been chosen to minimize exposure to secondhand smoke and to maintain the overall cleanliness of the premises.
* Smoking inside company buildings, offices, meeting rooms, and other enclosed spaces is strictly prohibited. This includes electronic cigarettes and vaping devices.
* All employees and visitors must adhere to relevant federa

In [27]:
agentFunction("Can I smoke in company vehicles?")

Question:  Can I smoke in company vehicles?
Answer:  Yes, you can find the information about smoking in company vehicles on page 5 of the provided context. According to that policy, "No Smoking in Company Vehicles" is stated as a restriction where smoking is not permitted in company vehicles, whether owned or leased.



 ------------ RESULT OBJECT ---------------

{'answer': 'Yes, you can find the information about smoking in company '
           'vehicles on page 5 of the provided context. According to that '
           'policy, "No Smoking in Company Vehicles" is stated as a '
           'restriction where smoking is not permitted in company vehicles, '
           'whether owned or leased.'}
